# 10.2 — Quantificatori (approfondimento)

**Prerequisiti:** 0.17 Quantificatori ∀ e ∃, 10.1 Logica proposizionale (approfondimento).


## Perché ci interessa

Nel notebook 0.17 abbiamo incontrato i quantificatori $\forall$ ("per ogni") ed $\exists$ ("esiste") come abbreviazioni comode. Ma nei prossimi notebook li useremo per scrivere definizioni con **più quantificatori annidati**: la definizione di limite (notebook 10.8) sarà della forma "per ogni $\varepsilon>0$ esiste $\delta>0$ tale che...", e quella di continuità uniforme (notebook 10.13) aggiungerà un ulteriore livello di annidamento. In queste definizioni, **l'ordine dei quantificatori cambia completamente il significato** della proposizione — e saperle negare correttamente è indispensabile per dimostrare che un limite *non* esiste o che una funzione *non* è uniformemente continua. Questo notebook costruisce lo strumento con cura, prima di usarlo sotto pressione.

## Teoria, passo dopo passo

### L'ordine dei quantificatori conta

Confrontiamo due proposizioni sui numeri reali, apparentemente simili:

$$\text{(A)}\quad \forall x\in\mathbb R\,\exists y\in\mathbb R: y>x \qquad\qquad \text{(B)}\quad \exists y\in\mathbb R\,\forall x\in\mathbb R: y>x$$

La (A) dice: "per ogni $x$, esiste un $y$ più grande di $x$" — vera (basta prendere $y=x+1$, e $y$ **può dipendere** da $x$). La (B) dice: "esiste un $y$ che è più grande di ogni $x$" — falsa (nessun numero reale è più grande di *tutti* gli altri, perché $\mathbb R$ non è limitato superiormente, notebook 5.2). La differenza è tutta nell'ordine: in (A) la scelta di $y$ può dipendere da $x$ (perché $x$ "viene prima"), in (B) $y$ deve andare bene per **tutti** gli $x$ contemporaneamente, essendo scelto prima di loro. Questa distinzione tra "il testimone può dipendere dalla variabile precedente" e "il testimone deve essere universale" è esattamente ciò che distingue la continuità puntuale dalla continuità uniforme (notebook 10.12 e 10.13): lì, il ruolo di $y$ sarà giocato da $\delta$.

### Negare i quantificatori

Come si nega $\forall x\, P(x)$ ("per ogni $x$, vale $P(x)$")? Dire che *non* è vero che $P(x)$ vale per ogni $x$ equivale a dire che **esiste** almeno un $x$ per cui $P(x)$ è falsa:

$$\neg(\forall x\, P(x)) \Leftrightarrow \exists x\, \neg P(x)$$

Simmetricamente, negare $\exists x\, P(x)$ ("esiste $x$ tale che $P(x)$") significa dire che **nessun** $x$ soddisfa $P(x)$, cioè che $P(x)$ è falsa per **tutti** gli $x$:

$$\neg(\exists x\, P(x)) \Leftrightarrow \forall x\, \neg P(x)$$

**In sintesi: negare un quantificatore lo scambia con l'altro, e sposta la negazione dentro, sul predicato.** Questa regola è l'analogo, per i quantificatori, delle leggi di De Morgan viste nel notebook 10.1 per $\wedge$ e $\vee$ — non a caso: $\forall$ generalizza una lunga congiunzione ("$P$ vale per $x_1$ **e** per $x_2$ **e** ..."), $\exists$ generalizza una lunga disgiunzione ("$P$ vale per $x_1$ **o** per $x_2$ **o** ...").

### Negare quantificatori annidati: un algoritmo

Per negare una proposizione con più quantificatori annidati, si applica la regola **ripetutamente**, attraversando la formula da sinistra a destra, scambiando ogni $\forall$ in $\exists$ (e viceversa) e spingendo la negazione sempre più a destra, fino a raggiungere il predicato finale — dove si applicano le regole del notebook 10.1 (De Morgan, negazione dell'implicazione).

**Esempio**: neghiamo $\forall\varepsilon>0\,\exists\delta>0\,\forall x\,(|x-x_0|<\delta\Rightarrow|f(x)-L|<\varepsilon)$ — la definizione di limite che vedremo nel notebook 10.8.

$$\neg\big[\forall\varepsilon\,\exists\delta\,\forall x\,(\ldots)\big] \;\Leftrightarrow\; \exists\varepsilon\,\forall\delta\,\exists x\,\neg(\ldots)$$

ogni $\forall$ diventa $\exists$ e viceversa, **nello stesso ordine relativo** in cui compaiono, con la negazione che alla fine colpisce solo il predicato più interno, dove si applica la regola di negazione dell'implicazione vista nel notebook 10.1: $\neg(P\Rightarrow Q)\Leftrightarrow P\wedge\neg Q$. Applicandola con $P=(|x-x_0|<\delta)$ e $Q=(|f(x)-L|<\varepsilon)$:

$$\exists\varepsilon>0\,\forall\delta>0\,\exists x\,\big(|x-x_0|<\delta \;\wedge\; |f(x)-L|\ge\varepsilon\big)$$

Questa è, parola per parola, la definizione rigorosa di "$f$ **non** tende a $L$ per $x\to x_0$" — la useremo esattamente in questa forma nel notebook 10.8.

In [ ]:
from itertools import product

# Verifichiamo su un dominio finito che negare i quantificatori funziona come descritto:
# la negazione di (per ogni x esiste y: P(x,y)) equivale a (esiste x per ogni y: not P(x,y))
dominio = range(-3, 4)

def P(x, y):
    return y > x  # esempio analogo alla proposizione (A)/(B) della teoria

# Valuta "per ogni x esiste y: P(x,y)" sul dominio finito
proposizione_A = all(any(P(x, y) for y in dominio) for x in dominio)
print("Proposizione A (per ogni x esiste y > x) sul dominio finito:", proposizione_A)

# La sua negazione, calcolata direttamente
negazione_diretta = not proposizione_A

# La negazione applicando la regola: esiste x per ogni y: not P(x,y)
negazione_per_regola = any(all(not P(x, y) for y in dominio) for x in dominio)

print("Negazione diretta:", negazione_diretta)
print("Negazione via regola (scambio quantificatori):", negazione_per_regola)
print("Coincidono?", negazione_diretta == negazione_per_regola)

## Esercizio svolto e commentato

**Problema:** nega la proposizione "ogni successione limitata ammette una sottosuccessione convergente" (anticipando il teorema di Bolzano-Weierstrass, già usato nel notebook 4.6 e che dimostreremo rigorosamente nel notebook 10.11), scritta in simboli come:

$$\forall (a_n)\text{ limitata}\,\exists (a_{n_k})\text{ sottosuccessione}\,\exists L\in\mathbb R:\; a_{n_k}\to L$$

**Ragionamento:**

**Passo 1 — identifichiamo i tre quantificatori annidati**: $\forall(a_n)$, poi $\exists(a_{n_k})$, poi $\exists L$, seguiti dal predicato "$a_{n_k}\to L$".

**Passo 2 — applichiamo la regola di scambio a ciascuno**, nello stesso ordine: $\forall\to\exists$, $\exists\to\forall$, $\exists\to\forall$:

$$\exists(a_n)\text{ limitata}\,\forall(a_{n_k})\text{ sottosuccessione}\,\forall L\in\mathbb R:\; \neg(a_{n_k}\to L)$$

**Passo 3 — traduciamo in italiano**: "esiste una successione limitata tale che, **per ogni** sua sottosuccessione e **per ogni** numero reale $L$, la sottosuccessione non converge a $L$" — cioè, più semplicemente, "esiste una successione limitata **nessuna** delle cui sottosuccessioni converge".

**Conclusione:** questa è l'affermazione che dovremmo dimostrare falsa se volessimo negare Bolzano-Weierstrass. Il teorema (dimostrato nel notebook 10.11) garantisce che una tale successione **non esiste** — ogni successione limitata ha sempre almeno una sottosuccessione convergente. Il valore di questo esercizio non è tanto il risultato, quanto il metodo: sapere scrivere con precisione **cosa dovremmo trovare** per confutare un teorema è il primo passo per apprezzarne la dimostrazione.

In [ ]:
# Illustriamo il fenomeno su un esempio concreto: ogni successione limitata (qui, periodica)
# ha effettivamente una sottosuccessione convergente -- coerente con Bolzano-Weierstrass.
import numpy as np

n = np.arange(1, 200)
a = (-1.0)**n + 1.0/n  # oscilla tra circa -1 e +1, limitata

# Sottosuccessione degli indici pari: converge a 1
sotto_pari = a[::2]
# Sottosuccessione degli indici dispari: converge a -1
sotto_dispari = a[1::2]

print("Ultimi termini della sottosuccessione pari (attesi vicino a 1):", sotto_pari[-5:])
print("Ultimi termini della sottosuccessione dispari (attesi vicino a -1):", sotto_dispari[-5:])
print("\nLa successione (a_n) non converge (oscilla), ma HA sottosuccessioni convergenti:")
print("questo è esattamente ciò che Bolzano-Weierstrass garantisce sempre, per ogni successione limitata.")

## Esercizi da risolvere

1. Nega la proposizione "esiste $M>0$ tale che per ogni $n\in\mathbb N$, $|a_n|\le M$" (la definizione di successione limitata).
2. Nega la proposizione "per ogni $\varepsilon>0$ esiste $N$ tale che per ogni $n>N$, $|a_n-L|<\varepsilon$" (anticipando la definizione di limite di successione, notebook 10.10).
3. Scrivi in simboli, con quantificatori, la proposizione "la funzione $f$ è limitata superiormente su $[a,b]$", poi negala.
4. *(più impegnativo)* Le proposizioni $\forall x\,\exists y\,P(x,y)$ e $\exists y\,\forall x\,P(x,y)$ non sono equivalenti in generale (lo abbiamo visto nella teoria). Dimostra però che la seconda **implica** la prima (cioè: se un singolo $y$ va bene per tutti gli $x$, allora per ogni $x$ esiste certamente un $y$ che va bene — lo stesso).

<details>
<summary><strong>👉 Soluzioni (clicca per visualizzare)</strong></summary>

**1.** $\neg\big[\exists M>0\,\forall n\,(|a_n|\le M)\big]\Leftrightarrow\forall M>0\,\exists n\,(|a_n|>M)$ — "per ogni $M$, esiste un termine della successione che supera $M$ in valore assoluto": la definizione di successione **non** limitata.

**2.** $\neg\big[\forall\varepsilon\,\exists N\,\forall n>N\,(|a_n-L|<\varepsilon)\big]\Leftrightarrow\exists\varepsilon>0\,\forall N\,\exists n>N\,(|a_n-L|\ge\varepsilon)$ — "esiste un $\varepsilon$ tale che, per quanto grande si scelga $N$, si trova sempre un termine oltre $N$ che resta a distanza almeno $\varepsilon$ da $L$": la definizione di "$a_n$ non tende a $L$".

**3.** "$f$ è limitata superiormente su $[a,b]$": $\exists M\in\mathbb R\,\forall x\in[a,b]\,(f(x)\le M)$. Negazione: $\forall M\in\mathbb R\,\exists x\in[a,b]\,(f(x)>M)$ — "per ogni $M$ esiste un punto dove $f$ supera $M$": $f$ non è limitata superiormente.

**4.** Supponiamo che valga $\exists y\,\forall x\,P(x,y)$: esiste un $\bar y$ tale che $P(x,\bar y)$ vale per ogni $x$. Vogliamo dimostrare $\forall x\,\exists y\,P(x,y)$: fissato un $x$ qualunque, dobbiamo esibire un $y$ per cui $P(x,y)$ è vera. Basta scegliere $y=\bar y$: per ipotesi $P(x,\bar y)$ vale (dato che $\bar y$ funziona per **ogni** $x$, in particolare per quello fissato). Quindi $\exists y\,P(x,y)$ è vera per ogni $x$, cioè vale $\forall x\,\exists y\,P(x,y)$. L'implicazione inversa non vale, come mostrato dal controesempio (A)/(B) nella teoria.

Verifica computazionale (esercizio 4, su un dominio finito):
```python
dominio = range(-3, 4)

def P(x, y):
    return y >= x  # esempio di predicato

esiste_y_universale = any(all(P(x, y) for x in dominio) for y in dominio)
per_ogni_x_esiste_y = all(any(P(x, y) for y in dominio) for x in dominio)

print("Esiste y universale => per ogni x esiste y:", (not esiste_y_universale) or per_ogni_x_esiste_y)
```

</details>

## Riepilogo finale

- L'**ordine dei quantificatori conta**: in $\forall x\,\exists y\,P(x,y)$ il testimone $y$ può dipendere da $x$; in $\exists y\,\forall x\,P(x,y)$ un solo $y$ deve valere per tutti gli $x$. La seconda implica la prima, ma non viceversa.
- Le regole di negazione $\neg(\forall x\,P)\Leftrightarrow\exists x\,\neg P$ e $\neg(\exists x\,P)\Leftrightarrow\forall x\,\neg P$ scambiano ogni quantificatore con l'altro, mantenendo l'ordine relativo.
- Per negare una proposizione con quantificatori annidati, si scambia **ogni** quantificatore, nell'ordine in cui compare, spingendo la negazione fino al predicato finale, dove si applicano le regole del notebook 10.1.
- Questo strumento sarà indispensabile a partire dal notebook 10.8, dove la definizione ε-δ di limite ha esattamente questa struttura annidata.

## Collegamenti

Il prossimo notebook è **10.3 — Insiemi, relazioni, funzioni (rigorosi)**: riprendiamo insiemi e funzioni con il rigore ora disponibile, definendo con precisione relazioni di equivalenza e d'ordine, in preparazione al Modulo 10.5 sulla cardinalità.